---
title: "River Flow for Nature vs Irrigation"
subtitle: "An environmental-flow rule in five TaqSim blocks, scored with fishy"
author:
  - name: Dr. Tobias Siegfried
    affiliation: hydrosolutions GmbH
date: today
format:
  html:
    theme: cosmo
    toc: true
    toc-depth: 2
    number-sections: true
    code-fold: false
    code-tools: true
    embed-resources: true
    include-in-header:
      - text: |
          <style>
          .cell-output-display img { max-width: 100%; height: auto; }
          main.content { max-width: 960px; }
          </style>
jupyter: python3
execute:
  echo: true
  warning: false
license: "CC BY 4.0"
---

# The question {#sec-question}

A river feeds a canal. The canal feeds the fields of Samarkand. Everything the canal does not take stays in the
river and flows on. The question for the operator of the headworks is not only *how much* water to leave in the
river, but *how* to leave it:

- **Leave a share.** Each day, a fixed share of the day's flow stays in the river, whatever the flow is.
- **Guarantee a minimum.** Each day, the first so-many cubic metres per second stay in the river; the canal takes
  only from what is above that.

Both can give the crops the same water over a year. They do not give the river the same regime. This notebook
builds the smallest TaqSim model that shows the difference, on the real Zarafshan at Ravatkhoja, and scores the
river with an index of hydrological alteration.

**What you will take away**

1. How TaqSim's environmental-flow rule (`EFlowSplit`) works, and how to wire it between a source, a canal and a sink.
2. How an alteration index (fishy's IARI, built on the 33 IHA indicators) turns a daily flow series into one number,
   and what that number leaves out.
3. Why, for the same water to the crops, a guaranteed minimum flow gives a river closer to natural than a share
   does: the shape of a rule matters, not only its volume.

> **The technical words.** The water left in the river for nature is the *environmental flow* (e-flow). The 33
> *Indicators of Hydrologic Alteration* (IHA; Richter et al. 1996) describe a flow regime by its magnitudes,
> timing, extremes, pulses and rates of change. The *IARI* (ISPRA 2011) compares those indicators between a
> natural and an altered series and averages the result: 0 means inside the natural range on every indicator.
> A plan that no other plan beats on both goals is *Pareto-optimal*; this notebook says "unbeaten".

# The system in five blocks {#sec-system}

TaqSim describes a water system as a network of blocks joined by edges. Ours has five blocks and five edges: an
inflow, two reaches that carry the rules, and two sinks.

In [ ]:
# ── Setup: run this cell first. In Colab it takes about a minute. ──────────────────────────────────
# It fetches the course folder, installs TaqSim, and tells Python where the course code lives.
import os
import sys
import subprocess

if not os.path.exists("src/zarafshan_taqsim"):  # not inside the course folder yet (the normal case in Colab)
    if not os.path.exists("zarafshan-taqsim-course"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/hydrosolutions/zarafshan-taqsim-course.git"], check=True)
    os.chdir("zarafshan-taqsim-course")
try:
    import fishy  # noqa: F401
except ImportError:
    %pip install -q polars "git+https://github.com/hydrosolutions/fishy.git@881660e8901987d63d89caf61ef0d30991000301"
sys.path[:0] = [os.path.abspath("eflow_game/src")]
print("Ready. Working folder:", os.getcwd())


In [ ]:
#| label: setup
# Standard tools
import inspect  # to show the source of a function inside this notebook
import json
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl  # tables; the new TaqSim and fishy both speak polars
from IPython.display import Markdown

# The game's reference model (workshop/eflow_game/src/eflow_game/model.py): the TaqSim system, the fishy scoring,
# the plan grid and the export. Everything the game page shows was computed by these functions.
from eflow_game import model as m
from eflow_game.model import Plan, build_system, demand_curve, load_inflow, run_year, score_river, twin_year

logging.basicConfig(level=logging.WARNING)
DAY = 86400.0

# The precomputed grid: every plan, run in TaqSim for 2019-2023 and scored with fishy
DATA = json.loads(m.DATA_FILE.read_text())
PLANS = pl.DataFrame(DATA["plans"]).select(
    "id", "name", "share_pct", "ceiling_m3s", "crops_pct", "score", "score_is_total", "scored", "classification", "unbeaten", "lowest_flow_m3s"
)
META = DATA["meta"]
Markdown(
    f"Loaded {len(DATA['plans'])} plans ({META['runs']} TaqSim runs of one year, {META['mean_run_seconds']:.1f} s each on "
    f"{META['workers']} processes, {META['wall_seconds_total'] / 60:.1f} min in all; built {META['built']})."
)

In [ ]:
#| label: fig-network
#| fig-cap: "The model as a network. Water flows from left to right. The two reaches carry the rules; the sinks only record what arrives."
#| echo: false
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.set_xlim(0, 10); ax.set_ylim(0, 3.2); ax.axis("off")
boxes = {
    "inflow": (0.9, 1.6, "Inflow\nSOURCE", "#4a6d8c"),
    "headworks": (3.3, 1.6, "Headworks\nREACH · EFlowSplit", "#0b0b0b"),
    "canal_gate": (5.8, 1.6, "Canal gate\nREACH · PriorityDistribution", "#0b0b0b"),
    "fields": (8.6, 2.5, "Fields\nSINK", "#007a00"),
    "river_end": (8.6, 0.7, "River end\nSINK", "#0f7f58"),
}
for key, (x, y, label, colour) in boxes.items():
    ax.add_patch(plt.Rectangle((x - 0.85, y - 0.42), 1.7 if key not in ("headworks", "canal_gate") else 2.0, 0.84, fc="white", ec=colour, lw=1.6))
    ax.text(x + (0 if key not in ("headworks", "canal_gate") else 0.15), y, label, ha="center", va="center", fontsize=9)
arrow = dict(arrowstyle="-|>", lw=1.8)
ax.annotate("", (2.45, 1.6), (1.75, 1.6), arrowprops=dict(color="#4a6d8c", **arrow))
ax.annotate("", (4.95, 1.6), (4.45, 1.6), arrowprops=dict(color="#4a6d8c", **arrow))
ax.annotate("", (7.75, 2.5), (7.0, 1.75), arrowprops=dict(color="#007a00", **arrow))
ax.annotate("", (7.75, 0.75), (7.0, 1.45), arrowprops=dict(color="#4a6d8c", **arrow))
ax.annotate("", (7.75, 0.6), (3.3, 1.18), arrowprops=dict(color="#0f7f58", connectionstyle="arc3,rad=0.25", **arrow))
ax.text(2.1, 1.75, "record", fontsize=8, ha="center", color="#52514e")
ax.text(4.7, 1.75, "flow − reserve", fontsize=8, ha="center", color="#52514e")
ax.text(7.5, 2.25, "demand of the day", fontsize=8, ha="center", color="#52514e")
ax.text(7.4, 1.0, "not needed", fontsize=8, ha="center", color="#52514e")
ax.text(5.0, 0.35, "reserve = min(share × flow, ceiling)", fontsize=8, ha="center", color="#0f7f58")
plt.show()

| Block | Kind | Rule | On the real river |
|:------|:-----|:-----|:------------------|
| Inflow | Source | passes on the recorded daily flow | the Zarafshan at the Ravatkhoja gauge |
| Headworks | Reach with `EFlowSplit` | reserves min(share × flow, ceiling) for the river, passes the rest on | the Ravatkhoja headworks |
| Canal gate | Reach with `PriorityDistribution` | gives the fields the day's demand, returns the rest to the river | the head of the Samarkand canals |
| Fields | Sink | records what arrives: the crops' water | Dargom, Mirzapay, Akkaradarya |
| River end | Sink | records what arrives: the river for nature | the Zarafshan below Ravatkhoja |

Nothing is stored and nothing is lost between the blocks, so on every day *inflow = river end + fields*. The model
runs day by day, one calendar year per run, and the five years 2019–2023 are scored together.

## The river {#sec-river}

In [ ]:
#| label: river
# The daily record at Ravatkhoja, 2010-2023, as the model reads it: columns date and q (m³/s)
inflow = load_inflow()
by_year = inflow.group_by(inflow["date"].dt.year().alias("year")).agg(pl.col("q").mean().round(1).alias("mean_m3s"), pl.col("q").min().alias("min_m3s"), pl.col("q").max().alias("max_m3s")).sort("year")
by_year

In [ ]:
#| label: fig-river
#| fig-cap: "The natural reference (grey, all 14 years) and the canal's demand curve (green). The five scored years are shaded."
#| echo: false
curve = np.array(demand_curve())
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(inflow["date"], inflow["q"], color="#878680", lw=0.7, label="Ravatkhoja record")
ax.axvspan(np.datetime64("2019-01-01"), np.datetime64("2023-12-31"), color="#0f7f58", alpha=0.08, label="scored years")
dates = inflow["date"].to_list()
ax.plot(dates, curve[[d.month - 1 for d in dates]], color="#007a00", lw=1.2, label="canal demand (monthly mean 2017–2022)")
ax.set_ylabel("m³/s"); ax.legend(loc="upper left", frameon=False); ax.set_xlim(dates[0], dates[-1])
plt.show()

Two facts decide everything that follows. The canal asks for little against the river: a mean of
`{python} f"{np.mean(curve):.1f}"` m³/s against a mean flow of `{python} f"{inflow['q'].mean():.0f}"` m³/s, and nothing
from November to February. And it asks at the wrong time for the river: the demand starts in March, when the river
is at its lowest of the year, and peaks in August, when the summer flood is already receding.

# The rule with two levers {#sec-rule}

TaqSim's `EFlowSplit` reserves an environmental flow before anything else is allocated. It has two parameters:

- `eflow_fraction`, the **reserve share** of the day's flow (0 to 1);
- `eflow_cap`, the **reserve ceiling**: the reserve never exceeds this volume per day.

The reserve of the day is *min(share × flow, ceiling)*. A share of 100 % with a ceiling of 60 m³/s therefore
means: the first 60 m³/s of every day stay in the river. A share of 50 % with no ceiling means: half of every day's
flow stays. The `PriorityDistribution` rule at the canal gate then serves the fields first, up to the month's
demand, and sends whatever remains back to the river.

In [ ]:
#| label: rule-source
# The function that wires the five blocks for one calendar year under one plan. Read it top to bottom:
# a time axis of daily steps, the inflow as an IntervalVolume, two sinks, the two reaches with their rules.
print(inspect.getsource(build_system))

A few plans in words, as the game names them:

In [ ]:
#| label: plan-names
for plan in (Plan(0, None), Plan(50, None), Plan(100, 60.0), Plan(50, 60.0), Plan(100, None)):
    print(f"share {plan.share_pct:3d} %, ceiling {'none' if plan.ceiling_m3s is None else f'{plan.ceiling_m3s:.0f} m³/s':10s} → {plan.name}")

# Build and run one year {#sec-run}

Let us run the driest of the scored years, 2021, under the plan "first 60 m³/s always stay". The run takes a few
seconds: TaqSim's new engine records every allocation it makes, so that the run can be replayed and audited.

In [ ]:
#| label: one-run
plan = Plan(100, 60.0)
run = run_year(2021, plan)  # a YearRun: dates, natural, river and fields in m³/s, and the run time
demand = m.demand_daily(run.dates)
crops_2021 = 100 * run.fields.sum() / demand.sum()
Markdown(
    m.engine_note() + f" Run time {run.seconds:.1f} s for {len(run.dates)} daily steps. Mass balance residual {run.mass_balance_residual_m3():.0f} m³. "
    f"Fields received {crops_2021:.1f} % of their demand. Lowest flow below the headworks {run.river.min():.1f} m³/s "
    f"(natural {run.natural.min():.1f})."
)

In [ ]:
#| label: fig-one-run
#| fig-cap: "2021 under “first 60 m³/s always stay”. The green band is the water the canal took."
#| echo: false
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(run.dates, run.natural, color="#878680", lw=1, ls="--", label="natural")
ax.plot(run.dates, run.river, color="#4a6d8c", lw=1.4, label="river below the headworks")
ax.fill_between(run.dates, run.river, run.natural, color="#007a00", alpha=0.3, label="taken by the canal")
ax.axhline(60, color="#0f7f58", lw=1, ls=":", label="guaranteed minimum 60 m³/s")
ax.set_ylabel("m³/s"); ax.legend(frameon=False, loc="upper left")
plt.show()

The same two rules fit in four lines of numpy. The model keeps this twin to check every TaqSim run: the two agree to
the engine's quantum of 1 m³ per day.

In [ ]:
#| label: twin
print(inspect.getsource(twin_year))
river_twin, fields_twin = twin_year(run.natural, demand, plan)
print("largest difference TaqSim − twin:", f"{np.abs(river_twin - run.river).max() * DAY:.2f} m³/day")

# Score the river {#sec-score}

The river score is fishy's IARI. It needs the 33 IHA indicators of the natural record, year by year, and the same
33 of the altered series over its last five years. For each indicator it asks: does the five-year value lie inside
the natural interquartile range? If yes, the indicator scores 0; if not, the distance outside, relative to the
range. The index is the mean of the 33. ISPRA's bounds: ≤ 0.05 close to natural, ≤ 0.15 good, above that altered.

Two things to say before the numbers. The method asks for at least 20 reference years; the Ravatkhoja record has
14 (2010–2023), and the project lead decided to use the available data, so the model patches fishy's check (see
`model._fishy`). And when fishy cannot score an indicator, it returns no total; the game then shows the mean of the
indicators it could score, says how many, and lists why.

In [ ]:
#| label: score-two-plans
# Score two plans over the five years 2019-2023. run_grid runs the years in a process pool (about ten seconds).
# The two example plans of the game page: the share-only and the minimum-only plan that the export paired
examples = [Plan(p["share_pct"], p["ceiling_m3s"]) for p in DATA["plans"] if p["id"] in META["teaching_pair"]]
runs = m.run_grid(examples, workers=5)
results = {p.id: m.evaluate_plan(p, runs[p.id]) for p in examples}
for r in results.values():
    s = r.river
    print(f"{r.plan.name:32s} crops {r.crops_pct:5.1f} %  score {s.score:.3f} ({'all 33' if s.total is not None else f'{s.scored} of 33'} indicators)  class {s.classification}")
    for w in r.warnings:
        print("   ⚠", w)

Nearly the same water for the crops; a different river. Which indicators make the difference?

In [ ]:
#| label: fig-indicators
#| fig-cap: "The 33 indicator scores of the two plans. Longer is further from the natural range."
#| echo: false
a, b = results.values()
names = [k for k in a.river.indicators]
fig, ax = plt.subplots(figsize=(10, 8))
y = np.arange(len(names))
ax.barh(y - 0.2, [a.river.indicators[k] or 0 for k in names], height=0.4, color="#1f66c1", label=a.plan.name)
ax.barh(y + 0.2, [b.river.indicators[k] or 0 for k in names], height=0.4, color="#0f7f58", label=b.plan.name)
ax.set_yticks(y); ax.set_yticklabels([k.replace("_", " ") for k in names], fontsize=8); ax.invert_yaxis()
ax.axvline(0.05, color="#878680", lw=0.8, ls=":"); ax.axvline(0.15, color="#878680", lw=0.8, ls=":")
ax.legend(frameon=False); ax.set_xlabel("indicator score (0 = inside the natural interquartile range)")
plt.show()

The share takes the most water, in absolute terms, on the days of highest flow, which barely matters to the index:
those days stay inside the natural range. But it also takes its share on the lowest days of spring, when the canal
has just started: the 1-, 3- and 7-day minima and the spring monthly flows move outside the natural range. The
guaranteed minimum takes nothing on those days. (Plan A can score only 32 of the 33 indicators, so its score is the
mean of 32; on those same 32, plan B's mean is given below.)

In [ ]:
#| label: common-indicators
common = [k for k, v in a.river.indicators.items() if v is not None and b.river.indicators[k] is not None]
for r in (a, b):
    print(f"{r.plan.name:32s} mean over the {len(common)} indicators both can score: {np.mean([r.river.indicators[k] for k in common]):.3f}")

# Every plan at once {#sec-grid}

The game page carries a grid of every lever combination: shares of 0–100 % in steps of 10, ceilings of 10, 20,
40, 60, 80, 100, 150 and 200 m³/s or none. A share of 0 % is the same plan whatever the ceiling, so it appears
once: `{python} len(DATA["plans"])` plans, each run in TaqSim for 2019–2023 and scored with fishy. Loading the
precomputed result instead of recomputing it keeps this notebook short.

In [ ]:
#| label: grid-table
PLANS.filter(pl.col("unbeaten")).sort("crops_pct", descending=True)

In [ ]:
#| label: fig-tradeoff
#| fig-cap: "Every plan: crops supplied against the river score. Filled dots are the plans no other plan beats. Right is better for the crops; down is better for the river."
#| echo: false
def family(row):
    if row["share_pct"] == 0 or (row["share_pct"] == 100 and row["ceiling_m3s"] is None):
        return "endpoints"
    if row["ceiling_m3s"] is None:
        return "leave a share"
    if row["share_pct"] == 100:
        return "guarantee a minimum"
    return "share with a ceiling"
colours = {"endpoints": "#0b0b0b", "leave a share": "#1f66c1", "guarantee a minimum": "#0f7f58", "share with a ceiling": "#9a9993"}
fig, ax = plt.subplots(figsize=(9, 5))
ax.axhspan(0, 0.05, color="#0f7f58", alpha=0.14); ax.axhspan(0.05, 0.15, color="#0f7f58", alpha=0.07)
for fam, colour in colours.items():
    rows = [r for r in DATA["plans"] if family(r) == fam]
    for r in rows:
        ax.scatter(r["crops_pct"], r["score"], s=46 if r["unbeaten"] else 26, facecolor=colour if r["unbeaten"] else "white", edgecolor=colour, lw=1.4, zorder=3 if r["unbeaten"] else 2)
    ax.scatter([], [], facecolor=colour, edgecolor=colour, label=fam)
pair = META["teaching_pair"]
for label, pid in zip("AB", pair):
    r = next(p for p in DATA["plans"] if p["id"] == pid)
    ax.annotate(label, (r["crops_pct"], r["score"]), xytext=(6, 6), textcoords="offset points", fontweight="bold")
ax.set_xlabel("crops supplied, % of the canal's demand 2019–2023"); ax.set_ylabel("river score (IARI), 0 = natural")
ax.legend(frameon=False, loc="upper left"); ax.set_xlim(-2, 102); ax.set_ylim(0, None)
plt.show()

# What to notice {#sec-notice}

In [ ]:
#| label: notice
#| echo: false
start = next(p for p in DATA["plans"] if p["id"] == META["starting_plan"])
A, B = (next(p for p in DATA["plans"] if p["id"] == pid) for pid in META["teaching_pair"])
unbeaten = [p for p in DATA["plans"] if p["unbeaten"]]
share_only = [p for p in DATA["plans"] if p["ceiling_m3s"] is None and 0 < p["share_pct"] < 100]
Markdown(f"""
1. **The canal takes little and still alters the river.** With the canal taking all it needs ("{start['name']}"),
   the crops get {start['crops_pct']:.0f} % and the river scores {start['score']:.3f}: {start['classification'] or 'altered'}, the mean of
   {start['scored']} of 33 indicators. The canal takes a few per cent of the year's water, but it takes it in
   spring and late summer, when the river is lowest.
2. **A share does little until it is large.** Of the share-only plans, {sum(p['crops_pct'] > 99.5 for p in share_only)} of
   {len(share_only)} change nothing for the crops, because the canal rarely needs more than what is left; and the score
   moves little with them.
3. **The same water, a different river.** Example A, "{A['name']}", gives the crops {A['crops_pct']:.1f} % at a score
   of {A['score']:.3f}; Example B, "{B['name']}", gives {B['crops_pct']:.1f} % at {B['score']:.3f}. The minimum
   protects the low-flow days; the share leaves them exposed.
4. **The unbeaten plans.** {len(unbeaten)} of {len(DATA['plans'])} plans are beaten by no other plan;
   {sum(p['share_pct'] == 100 and p['ceiling_m3s'] is not None for p in unbeaten)} of them are guaranteed minima.
   Choosing among them is a decision about how much river to trade for how much crop, not a calculation.
""")

# Limits {#sec-limits}

- **One canal.** At Ravatkhoja the real headworks also feed the transfers to Jizzakh and Kashkadarya and further
  canals; here only the three Samarkand canals divert, so the real canal share is larger and the real trade-off
  sharper.
- **No storage, losses or return flow.** The canal has no seepage, the fields drain nothing back, and nothing is
  carried from one day to the next. The game is about the headworks' rule, not the basin.
- **A fixed demand curve.** The canal asks for the same monthly volumes every year; a dry year does not change
  what the crops need.
- **The same rule every day.** Real environmental-flow rules vary by season and by type of year, and the
  literature designs them with many objectives at once (Suen & Eheart 2006; Chen & Olden 2017). Here the two levers
  are constant all year.
- **The reference is the recent regime.** The Zarafshan at Ravatkhoja is regulated upstream; 2010–2023 is the
  recent record, not a pristine river. And it has 14 years where the method asks for 20, so the natural quartiles
  are less certain than the method assumes, and the five scored years overlap the reference.
- **One index.** The IARI measures hydrological alteration; it says nothing about fish, temperature or sediment.

# Exercises {#sec-exercises}

1. Run `Plan(100, 40.0)` for 2021 with `run_year` and check the lowest flow below the headworks. Why is it not 40 m³/s?
   (Look at the natural minimum in @sec-river.)
2. Add a third rule family: a share that applies only from March to May. The `EFlowSplit` rule cannot do this by
   itself; sketch which block you would add, and what its rule would be.
3. Replace the demand curve with the six-canal total of the demand file (`m.SAMARKAND_CANALS` plus the other three
   columns) and recompute the two example plans. How does the trade-off change?
4. In the scored table, find the plan with the best river score among those that keep the crops above 95 %. Is it
   a share, a minimum, or a share with a ceiling?

# Sources {#sec-sources}

- Suen, J.-P. & Eheart, J. W. (2006). Reservoir management to balance ecosystem and human needs: incorporating the
  paradigm of the ecological flow regime. *Water Resources Research* 42, W03417.
- Chen, W. & Olden, J. D. (2017). Designing flows to resolve human and environmental water needs in a dam-regulated
  river. *Nature Communications* 8, 2158.
- Richter, B. D., Baumgartner, J. V., Powell, J. & Braun, D. P. (1996). A method for assessing hydrologic alteration
  within ecosystems. *Conservation Biology* 10, 1163–1174.
- ISPRA (2011). Implementazione della Direttiva 2000/60/CE: analisi e valutazione degli aspetti idromorfologici.
  Manuali e Linee Guida 131/2011. The IARI index and its class bounds, as implemented in fishy.
- hydrosolutions. *fishy*: environmental-flow diagnostics for TaqSim. github.com/hydrosolutions/fishy, commit 881660e.
- hydrosolutions. *TaqSim*. github.com/hydrosolutions/taqsim, commit 396ad09.
- Data: `data/ZRB_baseline/inflow/inflow_ravatkhoza_daily_2010_2023.csv` (daily flow at Ravatkhoja) and
  `data/ZRB_baseline/demand/demand_all_districts_2017-2022_monthly.csv` (monthly canal demands), both in the
  Zarafshan TaqSim repository.